# 第八课:LayerNorm——每个词自己的仪表校准

讲义:[08-layernorm.md](08-layernorm.md)。

In [ ]:
import numpy as np

def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def rms_norm(x, eps=1e-5):
    rms = np.sqrt((x**2).mean(axis=-1, keepdims=True) + eps)
    return x / rms

x = np.array([1., 2, 3, 4])
print('mu =', x.mean(), ' var =', x.var())
print('LN(x)      =', np.round(layer_norm(x), 4))
print('LN 后 mean =', layer_norm(x).mean(), ' std =', round(layer_norm(x).std(), 4))
print('RMSNorm(x) =', np.round(rms_norm(x), 4))

## 行与行互不干扰(讲义 §3)

In [ ]:
X = np.random.default_rng(0).normal(size=(3, 4)) * 5 + 2
ln = layer_norm(X)
print('各行均值:', np.round(ln.mean(-1), 8))
print('各行 std :', np.round(ln.std(-1), 4))

# 对照:轴放错(axis=0)会发生什么
wrong = (X - X.mean(0)) / np.sqrt(X.var(0) + 1e-5)
print('axis=0 版 各行均值:', np.round(wrong.mean(-1), 4), '← 不再是每行归零')

## 生产现场:padding 与 ε(作业 D3)

In [ ]:
z = np.zeros(4)
for eps in (0.0, 1e-8, 1e-5):
    try:
        out = z / np.sqrt((z**2).mean() + eps)
        print(f'eps={eps:g}: {out}')
    except Exception as e:
        print(f'eps={eps:g}: {type(e).__name__}: {e}')

## 作业现场

- C3:带 γ/β 的 layer_norm,先笔算 γ=2、β=1 的输出再验证。
- E1:观察 axis=0 的错误输出。

In [ ]:
# TODO(C3): 带参数版
def layer_norm_affine(x, gamma, beta, eps=1e-5):
    # 你的代码
    ...

# 自测:gamma=[2,2,2,2], beta=[1,1,1,1] 时,期望 = 2*LN(x)+1
# print(np.round(layer_norm_affine(x, np.full(4,2.), np.full(4,1.)), 4))